# NHANES 2017–2018 patient-need estimate

This notebook estimates a **cross-sectional CKD indicator**, not a confirmed chronic diagnosis. NHANES cannot establish persistence for at least three months. The primary adult (age 18+) rule is race-free 2021 CKD-EPI creatinine eGFR <60 mL/min/1.73 m² **OR** official NHANES UACR (URDACT) ≥30 mg/g and requires both defining measures. Known pregnant participants (RIDEXPRG=1) are excluded; negative, cannot-ascertain, and missing/not-applicable pregnancy codes remain eligible and are reported as such. Missing labs are unknown/excluded, never zero. eGFR-only and albuminuria-only sensitivity estimates use the same complete-case domain so their difference isolates the marker definition.

Population estimates use the NHANES 2-year MEC weight, masked variance strata and PSU, Taylor linearization, Student-t confidence limits, and full-sample domain handling. Raw data come only from the exact official CDC HTTPS files in the committed manifest and are never downloaded in CI.

In [1]:
import json
from dataclasses import asdict
from hashlib import sha256
from pathlib import Path

import pandas as pd

from ckd_intelligence.analysis.artifacts import patient_need_output_paths
from ckd_intelligence.cohorts.definitions import CKDDefinition, classify_ckd
from ckd_intelligence.statistics.survey import weighted_prevalence

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MANIFEST_PATH = ROOT / "data/manifests/nhanes-2017-2018-patient-need.json"
RAW_DIR = ROOT / "data/raw/nhanes/2017-2018"
FIXTURE_PATH = ROOT / "data/fixtures/nhanes_patient_need_representative.csv"
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

In [2]:
def checksum(path: Path) -> str:
    return sha256(path.read_bytes()).hexdigest()


expected = {item["name"]: item for item in manifest["files"]}
has_official_snapshot = all((RAW_DIR / name).is_file() for name in expected)
if has_official_snapshot:
    for name, metadata in expected.items():
        path = RAW_DIR / name
        if path.stat().st_size != metadata["bytes"]:
            raise ValueError(f"Unexpected byte count: {name}")
        if checksum(path) != metadata["sha256"]:
            raise ValueError(f"Checksum mismatch: {name}")
    demo = pd.read_sas(RAW_DIR / "DEMO_J.xpt", format="xport")[
        ["SEQN", "RIDSTATR", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "WTMEC2YR", "SDMVSTRA", "SDMVPSU"]
    ]
    biopro = pd.read_sas(RAW_DIR / "BIOPRO_J.xpt", format="xport")[["SEQN", "LBXSCR"]]
    albumin = pd.read_sas(RAW_DIR / "ALB_CR_J.xpt", format="xport")[
        ["SEQN", "URXUMA", "URXUCR", "URDACT"]
    ]
    frame = demo.merge(biopro, on="SEQN", how="left", validate="one_to_one").merge(
        albumin, on="SEQN", how="left", validate="one_to_one"
    )
    source_participant_count = len(frame)
    people = frame.rename(
        columns={
            "SEQN": "respondent_id",
            "RIDSTATR": "mec_exam_status",
            "RIDAGEYR": "age_years",
            "RIDEXPRG": "pregnancy_status_code",
            "LBXSCR": "serum_creatinine_mg_dl",
            "URXUMA": "urine_albumin_mg_l",
            "URXUCR": "urine_creatinine_mg_dl",
            "URDACT": "uacr_mg_g",
            "WTMEC2YR": "sample_weight",
            "SDMVSTRA": "strata",
            "SDMVPSU": "psu",
        }
    )
    people["sex"] = frame["RIAGENDR"].map({1.0: "Male", 2.0: "Female"})
    people.loc[people["sample_weight"].le(1e-12), "sample_weight"] = pd.NA
    people = people.loc[
        people["mec_exam_status"].eq(2)
        & people["sample_weight"].notna()
        & people["strata"].notna()
        & people["psu"].notna()
    ].copy()
    evidence_type = "public_observed"
    result_scope = (
        "NHANES 2017-2018 MEC-exam-weighted U.S. civilian, "
        "noninstitutionalized adults excluding known pregnant participants"
    )
else:
    people = pd.read_csv(FIXTURE_PATH)
    source_participant_count = len(people)
    evidence_type = "fixture_only"
    result_scope = "Representative fixture only; not a population estimate"

print({"evidence_type": evidence_type, "design_rows": len(people), "result_scope": result_scope})

{'evidence_type': 'public_observed', 'design_rows': 8704, 'result_scope': 'NHANES 2017-2018 MEC-exam-weighted U.S. civilian, noninstitutionalized adults excluding known pregnant participants'}


In [3]:
definitions = [CKDDefinition.primary(), CKDDefinition.egfr_only(), CKDDefinition.albuminuria_only()]
for definition in definitions:
    people[definition.name] = classify_ckd(people, definition)

adult = people["age_years"].ge(18)
known_pregnant = pd.to_numeric(people["pregnancy_status_code"], errors="coerce").eq(1).fillna(False)
eligible_domain = adult & ~known_pregnant
complete_primary = eligible_domain & people[CKDDefinition.primary().name].notna()
for definition in definitions:
    people.loc[~complete_primary, definition.name] = pd.NA
waterfall = [
    {
        "stage_order": 1,
        "stage": "NHANES 2017-2018 participants",
        "people": int(source_participant_count),
    },
    {
        "stage_order": 2,
        "stage": "MEC examined with valid design and positive weight",
        "people": int(len(people)),
    },
    {"stage_order": 3, "stage": "adults age 18+", "people": int(adult.sum())},
    {
        "stage_order": 4,
        "stage": "adults excluding known pregnancy",
        "people": int(eligible_domain.sum()),
    },
    {
        "stage_order": 5,
        "stage": "complete eGFR and UACR defining labs",
        "people": int(complete_primary.sum()),
    },
    {
        "stage_order": 6,
        "stage": "cross-sectional CKD indicator positive",
        "people": int(people.loc[complete_primary, CKDDefinition.primary().name].sum()),
    },
]
pd.DataFrame(waterfall)

,stage_order,stage,people
0,1,NHANES 2017-2018 participants,9254
1,2,MEC examined with valid design and positive we...,8704
2,3,adults age 18+,5533
3,4,adults excluding known pregnancy,5478
4,5,complete eGFR and UACR defining labs,5016
5,6,cross-sectional CKD indicator positive,938


In [4]:
source_population = result_scope
estimates = {}
for definition in definitions:
    estimate = weighted_prevalence(
        people[definition.name].tolist(),
        people["sample_weight"].tolist(),
        people["strata"].tolist(),
        people["psu"].tolist(),
        domain=eligible_domain.tolist(),
        lonely_psu="error",
        source_population=source_population,
    )
    values = people.loc[complete_primary, definition.name].astype(bool)
    record = asdict(estimate)
    record["unweighted_point"] = float(values.mean())
    estimates[definition.name] = record

pd.DataFrame(estimates).T[
    [
        "point",
        "standard_error",
        "ci_low",
        "ci_high",
        "denominator",
        "sum_weights",
        "degrees_freedom",
        "unweighted_point",
    ]
]

,point,standard_error,ci_low,ci_high,denominator,sum_weights,degrees_freedom,unweighted_point
primary_egfr_or_albuminuria,0.139189,0.006504,0.125326,0.153051,5016,227884099.89417,15,0.187002
sensitivity_egfr_only,0.05571,0.004963,0.045132,0.066289,5016,227884099.89417,15,0.081938
sensitivity_albuminuria_only,0.100024,0.004535,0.090359,0.10969,5016,227884099.89417,15,0.136563


In [5]:
primary = estimates["primary_egfr_or_albuminuria"]
artifact = {
    "analysis_id": "nhanes-2017-2018-cross-sectional-ckd-patient-need",
    "evidence_type": evidence_type,
    "result_scope": result_scope,
    "source_release": "2017-2018",
    "source_manifest": str(MANIFEST_PATH.relative_to(ROOT)),
    "input_provenance": manifest["files"]
    if has_official_snapshot
    else [{"name": str(FIXTURE_PATH.relative_to(ROOT)), "sha256": checksum(FIXTURE_PATH)}],
    "definition": {
        "population": "Adults age 18+ excluding known pregnancy, with complete defining labs",
        "primary": "Race-free CKD-EPI 2021 creatinine eGFR <60 mL/min/1.73 m2 OR UACR >=30 mg/g",
        "missing_labs": "Unknown and excluded; never false or zero",
        "interpretation": "Cross-sectional indicator, not confirmed chronic diagnosis",
        "pregnancy": "RIDEXPRG=1 excluded; codes 2/3 and missing/not-applicable retained",
        "uacr_source": "Official NHANES URDACT albumin creatinine ratio (mg/g)",
        "sensitivity": [
            "eGFR <60 only on common complete-case domain",
            "UACR >=30 mg/g only on common complete-case domain",
        ],
    },
    "survey_method": {
        "weight": "WTMEC2YR (2-year MEC exam weight)",
        "strata": "SDMVSTRA",
        "psu": "SDMVPSU",
        "variance": "Taylor linearization for a stratified PSU ratio",
        "confidence_interval": (
            "95% Student-t using PSU-minus-strata design degrees of freedom, bounded to [0,1]"
        ),
        "domain_handling": (
            "Full valid MEC-exam design retained; adult participants excluding "
            "known pregnancy form the domain; all definitions use the common "
            "complete-case subset"
        ),
        "lonely_psu": "error",
    },
    "waterfall": waterfall,
    "estimates": estimates,
    "finding": (
        "Among complete cases, weighted cross-sectional CKD-indicator "
        f"prevalence was {primary['point']:.1%} "
        f"(95% CI {primary['ci_low']:.1%}–{primary['ci_high']:.1%}; "
        f"n={primary['denominator']:,}) in {result_scope}."
    ),
    "limitations": [
        "One cross-sectional survey cycle cannot confirm chronicity or support diagnosis.",
        (
            "A single serum creatinine and spot UACR are subject to biological "
            "and measurement variability."
        ),
        (
            "The complete-case estimate excludes eligible adults missing either "
            "defining laboratory measure and may differ if their prevalence differs."
        ),
        (
            "Estimates describe the documented survey population and should not "
            "be interpreted as individual risk."
        ),
    ],
}
artifact_path, checksum_path = patient_need_output_paths(ROOT / "data", evidence_type)
artifact_path.parent.mkdir(parents=True, exist_ok=True)
artifact_bytes = (json.dumps(artifact, indent=2, sort_keys=True, allow_nan=False) + "\n").encode(
    "utf-8"
)
artifact_path.write_bytes(artifact_bytes)
artifact_digest = sha256(artifact_bytes).hexdigest()
checksum_path.write_text(
    f"{artifact_digest}  {artifact_path.relative_to(ROOT)}\n", encoding="utf-8"
)
print(artifact["finding"])
print(f"Artifact SHA-256: {artifact_digest}")

Among complete cases, weighted cross-sectional CKD-indicator prevalence was 13.9% (95% CI 12.5%–15.3%; n=5,016) in NHANES 2017-2018 MEC-exam-weighted U.S. civilian, noninstitutionalized adults excluding known pregnant participants.
Artifact SHA-256: e75517dcd31b632a32d43900e5bd26e1cbc731e9cc7db2500eec87bd13cb1356


## Interpretation guardrail

The estimate is a population-level screening/need signal for the documented NHANES sample. It is not a diagnosis, treatment recommendation, longitudinal progression estimate, or patient-targeting tool. Sensitivity differences show how much the definition depends on each available marker.